# Controllers Manual

Here we explain **every torque controller in the OpenExo firmware**: what kind of controller it is, what it is for, which parameters it takes, on which joints it can be used, and when you would use it.


**Where the code lives**

| What | File |
|---|---|
| Controller classes (the math) | `ExoCode/src/Controller.h`, `ExoCode/src/Controller.cpp` |
| Parameter index lists for each controller | `ExoCode/src/ControllerData.h` (namespace `controller_defs`) |
| Controller IDs per joint and their names in `config.ini` | `ExoCode/src/ParseIni.h` |
| Which joint can run which controller | `ExoCode/src/Joint.cpp` (`set_controller()` of each joint) |
| Parameter tables loaded at startup | `SDCard/<joint>Controllers/*.csv` |

The Python code cells only **illustrate** the torque profiles. We re-implement the firmware math in Python with example numbers, so you can see the shape of the curves and play with the parameters. (You need `numpy` and `matplotlib` to run them.)

> Everything in this notebook was read from the code at commit `523852a`. If the firmware changes, re-check the parameter tables against `ControllerData.h`.

## 1. Control basics you need for this notebook

**What a controller does here.** Every 2 ms (the main loop runs at 500 Hz) each joint asks its controller one question: *how much torque should the motor produce right now?* The controller answers with a number in **newton-metres (Nm)**. The joint divides it by the gear ratio and sends it to the motor driver, which handles the electrical current itself.

```
sensors (FSRs, angle, torque sensor, gait estimate)
        |
        v
  calc_motor_cmd()  ->  torque command [Nm]  ->  / gearing  ->  motor
```

**Feed-forward vs. feedback.** Almost every OpenExo controller has two parts.

* The **feed-forward** part (`ff_setpoint`, often called `cmd_ff` in the code) is the torque we *want*. It comes from a rule such as "push harder when the toe sensor reads more", or "follow this curve over the gait cycle".
* The **feedback** part is an optional **PID** correction. It compares the wanted torque with the torque actually measured by the torque sensor and adds a correction. Without it the system is **open loop**: we send the command and hope the motor delivers it. With it the system is **closed loop**.

The firmware combines them like this:

```
motor command = feed-forward + PID(feed-forward, measured torque)
```

**PID in one paragraph.** The error is `wanted - measured`.

* **P (proportional).** Corrects in proportion to the error now. More P gives faster tracking but can oscillate.
* **I (integral).** Corrects the error that has accumulated over time. It removes steady offsets but can wind up. The code comments say the team usually leaves it at 0.
* **D (derivative).** Reacts to how fast the measured torque changes. It damps oscillations. In OpenExo it uses the change of the *measurement*, not of the error, so a sudden change of setpoint does not cause a spike ("derivative on measurement").

Safety details of the shared `_pid()` function:

* If the torque sensor was never calibrated (`torque_offset_reading == 0`), PID is skipped and the feed-forward is sent alone.
* The I and D terms are only updated when the loop time is within 10 % of the expected 2 ms.
* The integral is reset when the trial is paused or not running.

**Filtering (EWMA).** Sensor signals are noisy. The code smooths them with an *exponentially weighted moving average*:

```
filtered = filtered + alpha * (new - filtered)
```

With `alpha = 1` there is no filtering. With a small `alpha` such as 0.05 the signal is very smooth but lags behind.

**Gait phase.** Walking controllers need to know *where in the step* the user is.

* **Percent gait** goes 0 to 100 % from one heel strike to the next. It is estimated from the duration of previous steps.
* **Percent stance** goes 0 to 100 % while the foot is on the ground and is 0 during swing.
* **FSRs** (force sensitive resistors) under the toe and heel tell whether the foot is loaded and how much. After calibration they are normalised so that about 1.0 means "normal full load".
* Several controllers offer a **simulated gait** flag. It replaces the real gait estimate with a fixed 1-second cycle, so you can test a torque profile on the bench without walking.

**Sign convention.** : **positive** is ankle dorsiflexion, knee extension, hip flexion and elbow flexion. Negative is ankle plantarflexion and hip extension. Motors can be flipped in `config.ini` (`flip_motor_dir`), so always verify directions on a new device with the Calibration Manager controller described below.

## 2. Overview: which controller runs on which joint

Choose a controller by its **`config.ini` name**. It is selected per joint, and can also be changed from the GUI or app during a trial. Every joint also has the option `0`, which disables the controller.

| Controller (class) | `config.ini` name | Hip | Knee | Ankle | Elbow | Arm 1 | Arm 2 | Type | Status |
|---|---|:-:|:-:|:-:|:-:|:-:|:-:|---|---|
| `ZeroTorque` | `zeroTorque` | ✓ | ✓ | ✓ | ✓ | ✓ | ✓ | Transparent / zero torque, optional PID | Stable |
| `ConstantTorque` | `constantTorque` | ✓ | ✓ | ✓ |  | ✓ | ✓ | Constant setpoint, optional PID | Stable |
| `Spline` | `spline` | ✓ |  | ✓ |  | ✓ | ✓ | Time-based profile with 5 nodes, optional PID | Stable |
| `Chirp` | `chirp` | ✓ | ✓ | ✓ | ✓ |  |  | Test signal: frequency sweep | Bench test |
| `Step` | `step` | ✓ | ✓ | ✓ | ✓ |  |  | Test signal: step pulses | Bench test |
| `CalibrManager` | `calibrManager` | ✓ | ✓ | ✓ | ✓ |  |  | Fixed-torque self-test | Setup tool |
| `ProportionalJointMoment` | `PJMC` |  |  | ✓ |  |  |  | Sensor-driven (toe FSR), PID with gain scheduling | Stable |
| `PJMC_PLUS` | `PJMC_PLUS` |  |  | ✓ |  |  |  | Sensor-driven, stateless, PID | Under development |
| `TREC` | `TREC` |  |  | ✓ |  |  |  | Sensor-driven + virtual spring/damper + propulsion, PD | Research |
| `SPV2` | `SPV2` |  |  | ✓ (right only) |  |  |  | Hybrid motor + servo-locked spring, PID, optimiser | Under development |
| `ZhangCollins` | `zhangCollins` |  |  | ✓ |  |  |  | Time-based profile (one bump), optional PID | Stable |
| `FranksCollinsHip` | `franksCollinsHip` | ✓ |  |  |  |  |  | Time-based profile (two bumps), optional PD | Stable |
| `ProportionalHipMoment` | `phmc` | ✓ |  |  |  |  |  | Sensor-driven state machine, open loop | Under development |
| `ElbowMinMax` | `elbowMinMax` |  |  |  | ✓ |  |  | Hand-FSR state machine, PID | Stable |
| `_cf_mfac` (helper) | none |  |  |  |  |  |  | Model-free adaptive control | Not used yet |

**Three families of controllers**

1. **Time-based (gait-phase) profiles**: `ZhangCollins`, `FranksCollinsHip`, `Spline`. The torque is a fixed curve plotted against percent gait or percent stance. They are simple and repeatable. However, they do not react if the user suddenly changes how they walk.
2. **Sensor-driven (proportional) controllers**: `PJMC`, `PJMC_PLUS`, `TREC`, `SPV2`, `PHMC`, `ElbowMinMax`. The torque follows a live sensor, usually the FSR load under the foot or in the hand. They adapt instantly to speed, slope and stairs.
3. **Utility and test controllers**: `ZeroTorque`, `ConstantTorque`, `Chirp`, `Step`, `CalibrManager`. These are used for setup, transparency, hardware validation and tuning.

**How the parameters reach the controller.** At startup each joint loads one row of the CSV file in `SDCard/<joint>Controllers/`. The row index is chosen in `config.ini` or from the GUI. **Columns are read by position**, not by header name. So the column order in the CSV must match the index order in `ControllerData.h`. The GUI and app can then overwrite single parameters live.

## 3.1 Zero Torque — `ZeroTorque` (`zeroTorque`)

**Type.** Constant zero setpoint with optional closed-loop PID. It is a "transparent mode" controller.

**Joints.** Hip, knee, ankle, elbow, arm 1 and arm 2 (all joints).

**What it does.** The wanted torque is always 0 Nm.

* **PID off.** The motor receives 0 Nm. The user still feels the friction and inertia of the motor and gearbox.
* **PID on** (requires a calibrated torque sensor). The controller measures the torque the user is putting into the device and drives the motor to cancel it. The device then feels lighter, almost "invisible". This is called *transparency*.

**Parameters** (`controller_defs::zero_torque`, file `zeroTorque.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `use_pid` | 1 = closed loop, 0 = open loop |
| 1 | `p_gain` | Proportional gain |
| 2 | `i_gain` | Integral gain (usually 0) |
| 3 | `d_gain` | Derivative gain |

**When to use it.**

* Baseline or "exo on, no assistance" condition in experiments.
* Letting the user walk or move with the device before turning assistance on.
* Checking that sensors and data logging work without applying torque.
* Tuning transparency gains before tuning an assistive controller.

**Tips.** If PID is on and the joint starts to buzz or oscillate, lower P or raise D slightly. If the torque sensor is not calibrated, PID is skipped automatically.

## 3.2 Constant Torque — `ConstantTorque` (`constantTorque`)

**Type.** Constant setpoint with optional closed-loop PID and an adaptive filter on the measured torque.

**Joints.** Hip, knee, ankle, arm 1 and arm 2. It is **not** available on the elbow.

**What it does.** It applies the same torque all the time.

* The `direction` flag picks the sign: 0 keeps the amplitude as is (+) and 1 makes it negative (−).
* While the toe FSR is being calibrated the output is forced to 0 Nm.
* When you change the amplitude, the torque-sensor filter is switched off (alpha = 1) until the measured torque reaches the new value. Then filtering resumes. This keeps the PID from reacting to a lagging filtered signal during the change.

**Parameters** (`controller_defs::constant_torque`, file `constantTorque.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `amplitude` | Torque magnitude in Nm |
| 1 | `direction` | 0 = positive sign, 1 = negative sign. Any other value leaves the sign as typed |
| 2 | `alpha` | Filter strength for the torque sensor, **in percent**. 100 means no filtering and 1 means very heavy filtering (alpha = 0.01) |
| 3 | `use_pid` | 1 = closed loop |
| 4–6 | `p_gain`, `i_gain`, `d_gain` | PID gains |

**When to use it.**

* Checking that a joint can hold a given torque, for example for heat or endurance tests.
* Static tests, and feeling how strong a torque level is before choosing setpoints.
* Simple resistance training, with a constant torque against the movement.

**Watch out.** The default CSVs for the hip and arms use `alpha = 1`. That means a filter alpha of 0.01, which is a very slow filter. The ankle and knee CSVs use 100 (no filter). Also, the code comment says direction 0 means "PF/Flexion", but at the ankle a positive torque is dorsiflexion in the sign convention. Check the real direction on your device before relying on it.

## 3.3 Spline — `Spline` (`spline`)

**Type.** Time-based (gait-phase) feed-forward profile with optional closed-loop PID.

**Joints.** Hip, ankle, arm 1 and arm 2. It is the same class everywhere. Each joint has its own CSV and its own parameter row.

**What it does.** You give it **five points** (x = percent of the cycle, y = torque in Nm). It draws a smooth curve through them, a *natural cubic spline*, and follows that curve as the cycle progresses.

* Before the first node the torque stays at node 1's value. After the last node it stays at node 5's value.
* The output is **clipped to ±15 Nm** for safety.
* If the x values are not strictly increasing, the output is 0 Nm. That is a safety fallback for bad parameters.

The "clock" for x can be one of three things:

* **percent stance** (`use_percent_gait = 0`, the legacy default). It is 0 during swing.
* **percent gait** (`use_percent_gait = 1`), from heel strike to heel strike.
* **simulated gait** (`sim_gait = 1`), a 1-second loop for bench tests. This overrides the other two.

**Parameters** (`controller_defs::spline`, file `spline.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0, 2, 4, 6, 8 | `nodeN_x` | Percent of the cycle (0–100) for nodes 1 to 5, must increase |
| 1, 3, 5, 7, 9 | `nodeN_y` | Torque in Nm at nodes 1 to 5 |
| 10 | `sim_gait` | 1 = use the simulated 1 s cycle |
| 11 | `use_percent_gait` | 1 = percent gait, 0 = percent stance |
| 12 | `use_pid` | 1 = closed loop |
| 13–15 | `p_gain`, `i_gain`, `d_gain` | PID gains |

**When to use it.**

* When you want **any shape** of assistance without writing new firmware, for example when trying a profile from a paper.
* Quick bench tests of a profile with `sim_gait = 1`.
* Arm joints, where there is no dedicated controller yet.

**Tip.** Natural cubic splines can **overshoot** between nodes when neighbouring y values change a lot. The cell below shows this. Always plot your nodes before sending them to the device.

## 3.4 Zhang–Collins — `ZhangCollins` (`zhangCollins`)

**Type.** Time-based (gait-phase) feed-forward profile with one torque "bump" and optional closed-loop PID.

**Joints.** Ankle only.

**Origin.** Zhang et al., *Human-in-the-loop optimization of exoskeleton assistance during walking*, Science, 2017. In that paper the ankle torque curve is described by four numbers (peak torque, peak time, rise time, fall time). An optimiser then searches for the values that minimise metabolic cost.

**What it does.** It draws one smooth bump against the gait clock:

* zero torque until `peak_time − rise_time`,
* a smooth rise to the peak torque at `peak_time`,
* a smooth fall back to zero at `peak_time + fall_time`,
* zero torque for the rest of the cycle.

The firmware output is **minus** the bump, `torque_cmd = −spline(...)`. With the PID flag on, a PID correction is added on top.

The clock is **percent stance**, not percent gait, even though the variable is called `percent_gait`. This comes from `_get_percent_gait(false)`, which returns percent stance, and it matches what the controller used before that helper existed. With `sim_gait = 1` a 1-second simulated cycle is used instead.

**Parameters** (`controller_defs::zhang_collins`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `torque` | Peak torque in Nm. The output sign is the opposite of this value |
| 1 | `peak_time` | Where the peak is, in % of the cycle |
| 2 | `rise_time` | Duration of the rise before the peak, in % |
| 3 | `fall_time` | Duration of the fall after the peak, in % |
| 4 | `direction` | Defined, but **not used** by the current code |
| 5 | `sim_gait` | 1 = simulated 1 s cycle |
| 6 | `use_pid` | 1 = closed loop |
| 7–9 | `p_gain`, `i_gain`, `d_gain` | PID gains |

**When to use it.**

* Plantarflexion push-off assistance during steady, level walking.
* Studies that compare against or reproduce the Zhang/Collins profile.
* Human-in-the-loop optimisation, because the shape is fully set by four numbers.

**⚠ Column order mismatch.** The header of `SDCard/ankleControllers/zhangCollins.csv` lists `Direction, use_pid, sim %gait`. The firmware reads index 5 as `sim_gait` and index 6 as `use_pid`. Columns are read by position, so the example row `-15,90,28,10,1,1,0,...` turns **simulated gait on and PID off**. That is the opposite of what the header suggests. Fix the CSV header or the column values before using those rows.

## 3.5 Franks–Collins Hip — `FranksCollinsHip` (`franksCollinsHip`)

**Type.** Time-based (gait-phase) feed-forward profile with **two** bumps and optional closed-loop **PD**. The I gain is ignored.

**Joints.** Hip only.

**Origin.** Franks et al., *Comparing optimized exoskeleton assistance of the hip, knee, and ankle in single and multi-joint configurations*, Wearable Technologies, 2021.

**What it does.** At the hip, walking needs **extension** torque (negative) in early stance and **flexion** torque (positive) around toe-off and early swing. The controller draws:

1. an extension bump (negative). It rises for `trough_onset` percent, peaks at `trough_percent_gait`, and ends where the zero-torque gap starts;
2. a zero-torque gap centred on `mid_time`, lasting `mid_duration`;
3. a flexion bump (positive) peaking at `peak_percent_gait`, lasting `peak_offset` after the peak.

Heel strike would split the curve in two, so the extension bump runs on a **shifted clock**. That clock restarts each time the gait estimate crosses `start_percent_gait`. The controller outputs **0 Nm until it has seen that first crossing**.

**Parameters** (`controller_defs::franks_collins_hip`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `mass` | User mass in kg. **Not used** at the moment |
| 1 | `trough_normalized_torque_Nm_kg` | Peak extension torque. Despite the name it is used **directly in Nm** |
| 2 | `peak_normalized_torque_Nm_kg` | Peak flexion torque, also in Nm |
| 3 | `start_percent_gait` | Where the shifted clock restarts |
| 4 | `trough_onset_percent_gait` | Duration of the rise into the extension trough |
| 5 | `trough_percent_gait` | Timing of peak extension |
| 6 | `mid_time` | Centre of the zero-torque gap between the bumps |
| 7 | `mid_duration` | Length of that gap |
| 8 | `peak_percent_gait` | Timing of peak flexion |
| 9 | `peak_offset_percent_gait` | Duration of the fall after peak flexion |
| 10 | `sim_gait` | 1 = simulated 1 s cycle |
| 11 | `use_pid` | 1 = closed loop (P and D only) |
| 12–14 | `p_gain`, `i_gain`, `d_gain` | Gains. `i_gain` is ignored |

**When to use it.** Hip assistance in steady walking: extension help in early stance and flexion help to swing the leg forward. It is also used in optimisation studies, since a few numbers describe the whole curve.


## 3.6 Proportional Joint Moment Control — `ProportionalJointMoment` (`PJMC`)

**Type.** Sensor-driven (proportional) feed-forward, plus closed-loop PID with optional **gain scheduling**.

**Joints.** Ankle only.

**Origin.** Gasparri, Luque & Lerner, *Proportional Joint-Moment Control for Instantaneously Adaptive Ankle Exoskeleton Assistance*, IEEE TNSRE, 2019.

**Main idea, in plain words.** The ankle muscles work hardest when the forefoot is pressed hardest. That moment is late stance, just before push-off. So the controller makes the exo torque **proportional to the toe FSR reading**. It needs no timing model. Walk faster or climb a slope, and the FSR reading changes, so the torque changes in the same step.

**How the torque is computed.**

* **Stance** (toe on the ground):
  `scaled = (min(toe_fsr, 1.2) − threshold) / (1 − threshold)`, then
  `cmd_ff = −max(0, scaled · stance_max)`, which is plantarflexion.
  With `is_assistance = 0` the sign is flipped, so it becomes **resistance** (dorsiflexion against the push).
* **Swing** (toe off the ground): `cmd_ff = swing_max`, a constant. A positive value helps lift the toe (dorsiflexion).
* **Closed loop.** The PID tracks `cmd_ff` with the filtered torque sensor reading.
* **Gain scheduling.** Near zero torque (setpoint within ±0.5 Nm *and* error within ±3.5 Nm) it switches to the softer gains `kp_zero/ki_zero/kd_zero`. This avoids buzzing during swing while keeping stiff tracking under load.

**Parameters** (`controller_defs::proportional_joint_moment`, file `PJMC.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `stance_max` | Peak plantarflexion torque in stance, in Nm |
| 1 | `swing_max` | Constant torque during swing, in Nm (sign as typed) |
| 2 | `is_assistance` | 1 = assist plantarflexion, 0 = resist |
| 3 | `use_pid` | 1 = closed loop |
| 4–6 | `p_gain`, `i_gain`, `d_gain` | Normal PID gains |
| 7 | `torque_alpha` | EWMA alpha for the torque sensor, from 0 to 1. A value of 0 means "use 0.5" |
| 8 | `GS_Flag` | 1 = gain scheduling on |
| 9–11 | `kp_zero`, `ki_zero`, `kd_zero` | Gains used near zero torque |

**When to use it.** This is the **default ankle assistance controller** for walking on varied terrain (level, slopes, stairs) and at changing speeds. It is also used for **resistance training** with `is_assistance = 0`.

**Notes.**

* The stance threshold is the toe FSR "upper threshold" from FSR calibration. Run the FSR calibration and refinement before using PJMC.
* The code also computes an adaptive gain `kf` per step, the ratio of peak setpoint to peak measured torque, clamped to 0.75–1.25. Right now `kf` is **only computed and not applied** to the command.

## 3.7 PJMC Plus — `PJMC_PLUS` (`PJMC_PLUS`)

**Type.** Sensor-driven, **stateless** proportional feed-forward with closed-loop PID. *Marked "under development" in the code.*

**Joints.** Ankle only.

**How it differs from PJMC.** PJMC first decides "stance or swing", then uses one rule for each. PJMC Plus uses **a single straight line** over the whole step, from the shared helper `_pjmc_generic()`:

* when the toe FSR is at or below `threshold`, the torque is `+dorsi_scaling` (dorsiflexion);
* the torque falls linearly as the toe FSR rises;
* at a toe FSR of 1.0 it reaches `−plantar_scaling` (plantarflexion);
* above 1.0 it keeps going linearly. The toe FSR is clipped at 1.5, so the plantarflexion can exceed the setpoint by up to about 50 %.

There is no stance/swing switch, so the transition from dorsiflexion to plantarflexion is continuous and has no jumps. PID runs whenever the torque sensor is calibrated.

**Parameters** (`controller_defs::pjmc_plus`, file `pjmc_plus.csv`). Only the parameters listed below are used. The others are kept so the CSV layout matches TREC and SPV2.

| Index | Name | Meaning |
|---|---|---|
| 0 | `plantar_scaling` | Plantarflexion torque at full toe load, in Nm |
| 1 | `dorsi_scaling` | Dorsiflexion torque when the toe is unloaded, in Nm |
| 2 | `timing_threshold` | Toe FSR level, **in %**, where the line starts |
| 7 | `kp` | P gain |
| 8 | `kd` | D gain |
| 11 | `ki` | I gain |
| 3–6, 9, 10, 12–16 | — | Not used by this controller |

**When to use it.** It is a smoother alternative to PJMC when you also want **dorsiflexion help in swing**, for example for foot drop, with no discontinuity at toe-off. It is also the "generic" term inside TREC and SPV2, so it is useful for tuning that term alone.

## 3.8 Terrain Responsive Exoskeleton Controller — `TREC` (`TREC`)

**Type.** Sensor-driven hybrid. It sums three feed-forward terms and closes the loop with **PD**. *Research controller.*

**Joints.** Ankle only.

**Origin.** C. F. Cuddeback, *Mixed Terrain Ankle Assistance and Modularity in Wearable Robotics* (NAU Biomechatronics Lab).

**The three terms.** Their sum is the feed-forward torque. The firmware flips the sign at the end, so plantarflexion is negative.

1. **Generic (PJMC-like) term.** This is the same straight line as PJMC Plus. It goes from `−dorsi_scaling` below the toe FSR threshold up to `+plantar_scaling` at full load. It provides the basic push-off.
2. **Suspension term (virtual spring + damper).** When the toe FSR first passes the threshold, the controller stores the current ankle angle as a *reference angle*. As the shin then rotates forward (dorsiflexion), a virtual spring pushes back:
   `k · (angle deviation) − b · angular velocity`. Only positive values are kept.
   A steep sigmoid on the toe FSR switches this term off when the foot is unloaded, and it is low-pass filtered. The term mimics the Achilles tendon storing energy, and it adapts to slopes. The "level entrance angle" captured during FSR refinement on level ground corrects the reference angle on inclines.
3. **Propulsive term.** `propulsive_gain · plantarflexion velocity`. It is only active near full toe load, gated by another sigmoid. It adds a push in the same direction the ankle is already moving during push-off.

**Extra features.**

* **Peak limiter** (`turn_on_peak_limiter = 1`). After each stance phase the controller compares the peak of *generic + suspension* to `plantar_scaling`. It then nudges the plantar setpoint up or down by 1 Nm, kept between 0 and 35 Nm, so the actual peak converges to the requested one.
* **Dynamic FSR threshold.** During swing the toe threshold is kept high until the heel is loaded. This avoids false triggers.
* **Safety.** The final command is limited to the feed-forward value ±35 Nm.

**Parameters** (`controller_defs::trec`, file `trec.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `plantar_scaling` | Target peak plantarflexion, in Nm |
| 1 | `dorsi_scaling` | Dorsiflexion torque when unloaded, in Nm |
| 2 | `timing_threshold` | Toe FSR threshold, in % |
| 3 | `spring_stiffness` | Virtual spring k. The code multiplies it by 0.01 |
| 4 | `neutral_angle` | Offset of the spring equilibrium. The code divides it by 100 |
| 5 | `damping` | Virtual damper b. The code multiplies it by 0.01 |
| 6 | `propulsive_gain` | Propulsive gain. The code multiplies it by 0.01 |
| 7 | `kp` | P gain |
| 8 | `kd` | D gain. There is no I term |
| 9 | `turn_on_peak_limiter` | 1 = adapt the plantar setpoint each step |

**When to use it.** Walking over **mixed terrain** (slopes, stairs, uneven ground) when you want more natural, tendon-like assistance than plain PJMC. The CSV has several preset rows of increasing strength, from 2 Nm up to 30 Nm.

**Notes.** The ankle angle is normalised, so stiffness and damping are in "Nm per % of range". For plotting, `desired_torque` is set to the propulsive term only, not to the total feed-forward. Look at `ff_setpoint` to see the full command.

## 3.9 SPV2 — `SPV2` (`SPV2`)

**Type.** Hybrid actuation: a sensor-driven motor command (PID) plus **servo-controlled mechanical parts** (a pawl lock and a leaf-spring stiffness adjuster), with an on-line optimiser. *Marked "still under development" in the code.*

**Joints.** Ankle, **right side only**. On the left side it always returns 0 Nm.

**What it does.**

* **Motor torque.** It uses the same generic line as PJMC Plus, from `+dorsi` to `−plantar` on the toe FSR, clamped to −45…+3 Nm. It is tracked with PID using **scaled** gains: P × 20 when the command is below −0.5 Nm (×10 otherwise), I × 80 and D × 20. An optional peak limiter adapts the plantar setpoint each step (+3 Nm or −1 Nm, kept between 0 and 65).
* **Pawl servo** (pin 27). When heel plus toe FSR exceeds `fsr_servo_threshold`, the servo moves to `servo_terminal` for 300 ms. This locks a mechanical spring so that it carries load. Otherwise the servo returns to `servo_origin`. While the pawl is engaged and the spring already gives more torque than asked, the motor is backed off to save battery.
* **Stiffness servo** (pin 26). In swing it sets the leaf-spring stiffness angle. The angle comes from `neutral_angle`, from the soft/stiff presets (`soft_or_stiff` = 1 or 2, or 10–20 to interpolate between them), or from the optimiser.
* **Optimiser** (`do_update_stiffness = 1`). Every `motor_current_calc_win` steps it measures average power and changes the stiffness angle to reduce it. Golden-section and simulated-annealing variants are also in the code. **The INA260 power-sensor code is commented out**, so the measured power is 0 and the optimiser does not move unless you re-enable that block.
* If the servo parameters are all zero (no SD row loaded) the controller outputs 0 Nm.

**Parameters** (`controller_defs::spv2`, 22 values, file `spv2.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `plantar_scaling` | Plantarflexion setpoint, in Nm |
| 1 | `dorsi_scaling` | Dorsiflexion setpoint, in Nm |
| 2 | `timing_threshold` | Toe FSR threshold, in % |
| 3 | `spring_stiffness_adj_factor` | Step size for stiffness changes, in degrees |
| 4 | `neutral_angle` | Starting stiffness servo angle, in degrees |
| 5, 6 | `min_angle`, `max_angle` | Limits of the stiffness servo, in degrees |
| 7, 8, 11 | `kp`, `kd`, `ki` | PID gains (scaled inside the code) |
| 9 | `turn_on_peak_limiter` | Adapt the plantar setpoint |
| 10 | `do_update_stiffness` | Run the stiffness optimiser |
| 12 | `do_use_servo` | Use the pawl servo |
| 13 | `fsr_servo_threshold` | FSR sum (in %) that triggers the pawl |
| 14, 15 | `servo_origin`, `servo_terminal` | Pawl servo home and engaged angles |
| 16 | `motor_current_calc_win` | Number of steps per optimiser evaluation |
| 17, 18 | `spring_stiffness`, `damping` | Defined, not used in the current code |
| 19 | `soft_or_stiff` | Stiffness preset selector |
| 20, 21 | `servo_angle_soft`, `servo_angle_stiff` | Servo angles for the soft and stiff presets |

**When to use it.** Only with the SPV2 hardware (motor plus servo-locked spring). It is a research platform for energy-efficient ankle assistance.

**⚠ CSV header mismatch.** The last columns in `spv2.csv` are labelled "TREC Spring, TREC Damper, TREC Propulsive, Do turn on the scanner, Do use stiffness control". The firmware reads them as indices 17–21, listed above. Check the values against the table before using a row.

## 3.10 Proportional Hip Moment Controller — `ProportionalHipMoment` (`phmc`)

**Type.** Sensor-driven **state machine**, **open loop** (no PID). *Marked "still under development" in the code.*

**Joints.** Hip only.

**Main idea.** It applies the PJMC idea to the hip. It estimates the hip moment from the **balance between heel and toe load**:

`fs = heel_fsr − 0.25 · toe_fsr`

Heel loading (early stance) means the hip needs **extension**. Toe loading (late stance) means it is time to prepare **flexion**.

**The three states.**

| State | Phase | Torque rule |
|---|---|---|
| 2 | **Stance** (foot loaded) | Proportional to `fs`. Positive `fs` gives extension and negative `fs` gives flexion. A smooth polynomial blends late stance into swing |
| 3 | **Early swing** (first ~15 % of swing) | It continues the flexion curve. Timing is scaled by the duration of the previous step |
| 1 | **Late swing** | The torque ramps linearly toward 60 % of the extension setpoint, ready for the next heel strike |

Each transition needs **3 consecutive loop iterations** that agree, to reject sensor noise. If the computed torque ever exceeds 20 Nm in magnitude, it is set to 0 as a safety net.

**Parameters** (`controller_defs::proportional_hip_moment`, file `PHMC.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `extension_setpoint` | Scale of the extension torque, in Nm |
| 1 | `flexion_setpoint` | Scale of the flexion torque, in Nm |

**When to use it.** Experimental hip assistance that adapts to the user's own gait without a fixed time profile. It runs open loop, so it works on hips **without a torque sensor**. The command is not corrected, though.

**Notes.** Timings are counted in **loop iterations**, not seconds, so they assume the 500 Hz loop. The state and `fs` are stored in the controller data so you can plot them.

## 3.11 Elbow Min–Max — `ElbowMinMax` (`elbowMinMax`)

**Type.** Sensor-driven **state machine** with hysteresis, an optional angle-dependent "spring" profile, a smoothed setpoint and closed-loop **PID** (always on).

**Joints.** Elbow only.

**Origin.** Colley, Bowersock & Lerner, *A Lightweight Powered Elbow Exoskeleton for Manual Handling Tasks*, IEEE T-MRB, 2024.

**What it does.** The user wears a glove with two FSRs. The firmware reuses the foot channels for them: the **digit/grip** sensor is read on the "toe" channel and the **palm** sensor on the "heel" channel. The elbow angle is read from the "ankle" angle slot.

1. **Calibration** normalises both FSRs to 0–1.
   * `CaliRequest = 3`: use the stored "factory" min/max values for the left or right glove. This runs once at startup.
   * `CaliRequest = 1`: a 10-second manual calibration. Squeeze and release while it records min/max of the FSRs and the elbow angle.
2. **State detection**, with thresholds multiplied by 0.05 so a value of 4 means 0.2 normalised:
   * **Flex**: the grip sensor is above its upper threshold and higher than the palm sensor. The setpoint becomes `+FLEXamplitude`, helping to lift.
   * **Extend**: the palm sensor is above its upper threshold. The setpoint becomes `−EXTamplitude`, helping to push or lower.
   * **Null**: both sensors are below their *lower* thresholds. The setpoint becomes 0.
   * Otherwise the previous setpoint is kept. The gap between upper and lower thresholds is the **hysteresis**: it stops the torque from flickering on and off.
3. **Spring profile** (`TrqProfile = 1`). A polynomial in the normalised elbow angle, scaled by `SpringPkTorque`, is added to the setpoint. Torque then changes with the arm position, like a spring.
4. **Smoothing and limits.** The setpoint is filtered with alpha = `FiltStrength/100`. Lower values give a softer, slower torque rise and are more comfortable. The setpoint is then clipped to ±`TorqueLimit`, and tracked with PID.

**Parameters** (`controller_defs::elbow_min_max`, file `elbowMinMax.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `FLEXamplitude` | Flexion torque setpoint, in Nm |
| 1 | `DigitFSR_threshold` | Grip upper threshold (×0.05) |
| 2 | `PalmFSR_threshold` | Palm upper threshold (×0.05) |
| 3 | `DigitFSR_LOWthreshold` | Grip lower threshold (×0.05) |
| 4 | `PalmFSR_LOWthreshold` | Palm lower threshold (×0.05) |
| 5 | `CaliRequest` | 1 = manual 10 s calibration, 3 = factory values |
| 6 | `TrqProfile` | 1 = spring profile, 0 = constant |
| 7–9 | `P_gain`, `I_gain`, `D_gain` | PID gains |
| 10 | `TorqueLimit` | Maximum absolute setpoint, in Nm |
| 11 | `SpringPkTorque` | Scale of the spring profile, in Nm |
| 12 | `EXTamplitude` | Extension torque setpoint, in Nm |
| 13 | `FiltStrength` | Setpoint smoothing, in % (alpha = value / 100) |

**When to use it.** Lifting, carrying and other manual handling tasks with the elbow exoskeleton. The user "asks" for help simply by gripping an object or pressing with the palm.

## 3.12 Chirp — `Chirp` (`chirp`)

**Type.** Test-signal generator: a sine wave with a rising frequency (a *chirp*), with optional closed-loop PID.

**Joints.** Hip, knee, ankle and elbow. **Only the left side is driven.** The right side always returns 0 Nm.

**What it does.** For `t` from 0 to `duration` seconds:

```
f(t)   = start_freq + (end_freq − start_freq) · t / duration
torque = amplitude · sin(2π · f(t) · t) + yshift
```

After `duration` the output is 0 Nm. To **restart** the sweep without changing controller, set the amplitude to 0 and then back to the wanted value.

**Parameters** (`controller_defs::chirp`, file `chirp.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `amplitude` | Sine amplitude, in Nm |
| 1 | `start_frequency` | Start frequency, in Hz |
| 2 | `end_frequency` | End frequency, in Hz |
| 3 | `duration` | Length of the sweep, in s |
| 4 | `yshift` | Constant offset added to the sine, in Nm |
| 5 | `pid_flag` | 1 = closed loop |
| 6–8 | `p_gain`, `i_gain`, `d_gain` | PID gains |

**When to use it.** **Hardware validation and bandwidth testing**, with the device on the bench or worn but relaxed. Compare `ff_setpoint` with the measured torque. The frequency at which the measured torque gets clearly smaller or lags behind shows how fast the actuator can follow. That limit tells you how aggressive your assistive profiles and PID gains can be.

**Note.** The phase is computed as `f(t) · t`, so the *real* instantaneous frequency rises twice as fast as `f(t)`. It ends near `2·end_freq − start_freq`, not at `end_freq`. Keep this in mind when you read frequency plots. The next cell shows the effect.

## 3.13 Step — `Step` (`step`)

**Type.** Test-signal generator: a train of rectangular torque pulses, with optional closed-loop PID.

**Joints.** Hip, knee, ankle and elbow.

**What it does.** During an active, unpaused trial it repeats `repetitions` times:

1. hold `amplitude` Nm for `duration` seconds;
2. hold 0 Nm for `spacing` seconds.

After the last repetition the output stays at 0. Pausing or ending the trial resets the counter, the timers and the PID integral, so the next trial starts the sequence again.

**Parameters** (`controller_defs::step`, file `step.csv`):

| Index | Name | Meaning |
|---|---|---|
| 0 | `amplitude` | Pulse torque, in Nm (sign as typed) |
| 1 | `duration` | Pulse length, in s |
| 2 | `repetitions` | Number of pulses |
| 3 | `spacing` | Pause between pulses, in s |
| 4 | `pid_flag` | 1 = closed loop |
| 5–7 | `p_gain`, `i_gain`, `d_gain` | PID gains |
| 8 | `alpha` | Torque sensor filter, in % (100 = no filter) |

**When to use it.**

* Measuring the **step response**: rise time, overshoot and settling time. These are the classic numbers for **tuning PID gains**. Try a gain set, look at the response, adjust, and repeat.
* **Maximum torque capacity** tests, and checking that the torque sensor reads what the motor delivers.

## 3.14 Calibration Manager — `CalibrManager` (`calibrManager`)

**Type.** Self-test "controller": a fixed, open-loop torque with printouts to the serial port.

**Joints.** Hip, knee, ankle and elbow.

**What it does.** During an active, unpaused trial it commands **3.5 Nm** in the positive direction. For Maxon motors it sends a raw command of 100 instead, because their command is not converted through the torque constant. When the trial is not active, or is paused, it commands 0. About once per second it prints to the serial monitor:

```
Left angle: ...  |  Left torque: ...  |  Left cmd: 3.5
```

It does **not** use PID. It has one parameter, `calibr_cmd`, which is not used. No CSV file is needed.

**When to use it.** **Always run it first on a new or rebuilt device**, especially one with a torque sensor.

* Check that a positive command moves the joint in the expected *positive* direction: dorsiflexion, knee extension, hip or elbow flexion. If not, fix `flip_motor_dir` in `config.ini`.
* Check that the torque sensor reads a **positive** value when the motor pushes in the positive direction. If the motor and sensor signs disagree, a closed-loop PID **amplifies** the error instead of correcting it. That can be dangerous.

> The printed "angle" always comes from the ankle angle slot, even when the controller runs on another joint.

## 4. Choosing a controller — quick guide

| I want to… | Use |
|---|---|
| Check a brand-new device's directions and torque-sensor sign | `calibrManager` |
| Let the user wear the device without assistance (baseline) | `zeroTorque`, with PID for a lighter feel |
| Find out how fast my actuator can follow a command | `chirp` |
| Tune PID gains, or test maximum torque | `step` |
| Apply a steady torque | `constantTorque` |
| Assist the ankle in daily walking, on any terrain or speed | `PJMC` |
| Assist the ankle with smooth dorsi-to-plantar help, for example for foot drop | `PJMC_PLUS` |
| Tendon-like ankle help on slopes and stairs | `TREC` |
| Reproduce Zhang/Collins ankle assistance, or run optimisation studies | `zhangCollins` |
| Assist the hip in walking with a fixed profile | `franksCollinsHip` |
| Assist the hip without a torque sensor, adapting to gait | `phmc` (experimental) |
| Any custom profile on the hip, ankle or arm | `spline` |
| Help lifting with the elbow exo | `elbowMinMax` |

**General safety checklist before turning assistance on**

1. Run `calibrManager` once on new hardware and verify both signs.
2. Run **torque sensor calibration** and **FSR calibration and refinement** from the GUI. PID is disabled without torque calibration, and the FSR-based controllers need calibrated FSRs.
3. Start with small setpoints and the PID **off**, then turn the PID on with low P.
4. Plot `ff_setpoint` against the measured torque. If they do not match, tune the gains with `step` before walking.
5. Remember that CSV columns are read **by position**. Check every CSV row against the parameter table of its controller.